# RF De-embedding with Synthetic S-Parameters (Validation Project)

## Goal
This project studies **RF de-embedding**: removing measurement fixture effects from a composite two-port S-parameter measurement in order to recover the intrinsic S-parameters of the **Device Under Test (DUT)**.

## Why de-embedding?
A Vector Network Analyzer (VNA) measures S-parameters between calibrated reference planes. In real setups, the DUT is rarely connected directly at those planes; instead, fixtures (launches, traces, adapters, probes) sit between the VNA and the DUT. De-embedding is a post-processing step that mathematically removes the fixture response, leaving an estimate of the DUT-only response.

## What we implement
- A **forward model** that synthesizes realistic measurements (including noise).
- An **inverse model** (de-embedding) that estimates DUT-only behavor.
- A validation step comparing de-embedded results to a known, noise-free **golden** DUT model.

## Datasets produced by the generator
- `fixture.csv`           : fixture-only measurement (with noise)
- `dut_plus_fixture.csv`  : fixture + DUT + fixture measurement (with noise)
- `golden_dut.csv`        : DUT-only "ground truth" (no noise)


## S-parameters (two-port)

At each port we define:
- incident wave: $a_1, a_2$
- reflected wave: $b_1, b_2$

For a two-port network:

$$
\begin{bmatrix}
b_1 \\
b_2
\end{bmatrix}
=
\begin{bmatrix}
S_{11} & S_{12} \\
S_{21} & S_{22}
\end{bmatrix}
\begin{bmatrix}
a_1 \\
a_2
\end{bmatrix}
$$

Interpretation (with the *other* port matched so its incident wave is zero):
- $S_{11} = b_1/a_1$ with $a_2=0$ (input reflection)
- $S_{21} = b_2/a_1$ with $a_2=0$ (forward transmission)
- $S_{12} = b_1/a_2$ with $a_1=0$ (reverse transmission)
- $S_{22} = b_2/a_2$ with $a_1=0$ (output reflection)

We assume a fixed reference impedance $Z_0 = 50\,\Omega$.

## Cascades and the ABCD (chain) matrix

De-embedding is easiest when the measurement chain is expressed as cascaded two-port networks.

A common form is the **ABCD matrix**:

$$
\begin{bmatrix}
V_1 \\
I_1
\end{bmatrix}
=
\begin{bmatrix}
A & B \\
C & D
\end{bmatrix}
\begin{bmatrix}
V_2 \\
I_2
\end{bmatrix}
$$

Key property: if two two-ports are cascaded, their ABCD matrices multiply:

$$
M_{total} = M_1 \cdot M_2 \cdot M_3 \cdots
$$

This is why we often convert measured S-parameters to ABCD (or a related transfer matrix),
do the cascade algebra, and convert back to S-parameters.